# Multi-LLM Evaluator (Interactive)

Run each cell from top to bottom. Output appears after every step so you can read the question, review model answers as they arrive, and run the ranking only when you are ready.

**Setup:** Create a `.env` file in the project root with your API keys (see `.env.example`).

In [ ]:
import asyncio
import sys
from pathlib import Path

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "evaluator.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

from src.config import MODEL_REGISTRY, _env_value, get_gemini_api_key, _PROJECT_ROOT
from src.evaluator import MultiLLMEvaluator

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))

In [ ]:
checks = [
    ("OPENAI_API_KEY", "OpenAI", _env_value("OPENAI_API_KEY")),
    ("GEMINI_API_KEY", "Gemini", get_gemini_api_key()),
    ("GROQ_API_KEY", "Groq", _env_value("GROQ_API_KEY")),
]

missing = [name for name, _, value in checks if not value]
if missing:
    raise RuntimeError(
        f"Missing API keys: {', '.join(missing)}. "
        f"Set them in {_PROJECT_ROOT / '.env'}"
    )

display(Markdown("All API keys loaded."))
display(Markdown("**Models in this run:**\n\n" + "\n".join(
    f"- {cfg['display_name']} (`{model_id}`)" for model_id, cfg in MODEL_REGISTRY.items()
)))

In [ ]:
runner = MultiLLMEvaluator()
question = runner.generate_benchmark_question()

display(Markdown(f"## Evaluation Question\n\n{question}"))

In [ ]:
runner.responses = {}
display(Markdown("## Model Responses\n\nEach answer appears below as soon as its model finishes."))

tasks = [
    asyncio.create_task(runner.fetch_single_response(model_id, config, question))
    for model_id, config in MODEL_REGISTRY.items()
]

shown = set()
for finished in asyncio.as_completed(tasks):
    await finished
    for model_id, answer in runner.responses.items():
        if model_id in shown:
            continue
        shown.add(model_id)
        name = MODEL_REGISTRY[model_id]["display_name"]
        display(Markdown(f"### {name}\n\n{answer}"))

display(Markdown(f"**Done — {len(runner.responses)} / {len(MODEL_REGISTRY)} responses received.**"))

In [ ]:
display(Markdown("## All Responses\n\nReview each answer before running the judging step."))

for model_id, answer in runner.responses.items():
    name = MODEL_REGISTRY[model_id]["display_name"]
    display(Markdown(f"---\n\n### {name} (`{model_id}`)\n\n{answer}"))

In [ ]:
display(Markdown("## Judging"))
leaderboard = runner.judge_and_rank(question)

lines = ["## Final Leaderboard\n"]
for rank, model_id in enumerate(leaderboard, 1):
    name = MODEL_REGISTRY[model_id]["display_name"]
    lines.append(f"{rank}. **{name}** (`{model_id}`)")

display(Markdown("\n".join(lines)))